In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.linear_model import LogisticRegression

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

dirName = 'spatialTaskLateMapping'
activityFileName = 'activitityTestGrid.npz'

cwd = Path.cwd().resolve()
if cwd.name == 'Analysis':
    analysisRoot = cwd
    trainingRoot = cwd.parent / 'Training'
elif cwd.name == 'Training':
    trainingRoot = cwd
    analysisRoot = cwd.parent / 'Analysis'
else:
    trainingRoot = Path('/gpfsnyu/home/zl6041/DURF_project/EconomicDecisionMakingReferenceFrame/Training')
    analysisRoot = trainingRoot.parent / 'Analysis'

savedRoot = trainingRoot / 'savedForHPC'
figureRoot = analysisRoot / 'Figure'
figureRoot.mkdir(parents=True, exist_ok=True)

modelDirs = sorted([
    p for p in (savedRoot / dirName).iterdir()
    if p.is_dir() and (p / activityFileName).exists()
]) if (savedRoot / dirName).exists() else []

print('savedRoot:', savedRoot)
print('figureRoot:', figureRoot)
print('n spatialTaskLateMapping models:', len(modelDirs))
if modelDirs:
    print('example:', modelDirs[0])


In [ ]:
dt = 10
response_window = (3000, 3200)


def loc12_to_label(loc12):
    if isinstance(loc12, str):
        return loc12
    loc12 = np.asarray(loc12)
    return '12' if loc12[0] == 1 else '21'


def mean_response_output(model_output, trial_params, mask=None, dt=dt):
    if mask is not None and np.any(mask):
        mask_sum = np.sum(mask, axis=1)
        mask_sum[mask_sum == 0] = 1
        return np.sum(mask * model_output, axis=1) / mask_sum

    response_output = np.zeros((model_output.shape[0], model_output.shape[2]))
    for iTrial, params in enumerate(trial_params):
        response_onset = params.get('fixation_offset', response_window[0])
        response_offset = params.get('end', response_window[1])
        i0 = int(round(response_onset / dt))
        i1 = int(round(response_offset / dt))
        i0 = max(0, min(i0, model_output.shape[1] - 1))
        i1 = max(i0 + 1, min(i1, model_output.shape[1]))
        response_output[iTrial, :] = np.mean(model_output[iTrial, i0:i1, :], axis=0)
    return response_output


def juice_for_offer(seqAB, chosen_offer):
    if seqAB == 'AB':
        return 'A' if chosen_offer == 1 else 'B'
    return 'B' if chosen_offer == 1 else 'A'


def derive_choice_labels_from_model_output(model_output, trial_params, mask=None):
    response_output = mean_response_output(model_output, trial_params, mask=mask)
    choice_right = response_output[:, 1] > response_output[:, 0]
    choiceLR = np.array(['right' if right else 'left' for right in choice_right])

    seqAB = np.array([trial_params[i]['seqAB'] for i in range(len(trial_params))])
    loc12_label = np.array([
        trial_params[i].get('loc12_label', loc12_to_label(trial_params[i]['loc12']))
        for i in range(len(trial_params))
    ])

    chosen_offer = np.zeros(len(trial_params), dtype=int)
    for iTrial in range(len(trial_params)):
        offer1_left = loc12_label[iTrial] == '12'
        if choiceLR[iTrial] == 'left':
            chosen_offer[iTrial] = 1 if offer1_left else 2
        else:
            chosen_offer[iTrial] = 2 if offer1_left else 1

    choice12 = np.array(['2' if offer == 2 else '1' for offer in chosen_offer])
    choiceAB = np.array([
        juice_for_offer(seqAB[iTrial], chosen_offer[iTrial])
        for iTrial in range(len(trial_params))
    ])
    choiceB = np.array([juice == 'B' for juice in choiceAB])

    return choice12, choiceB, choiceLR, seqAB, loc12_label


def importAndPreprocess(dirPath, activityFileName=activityFileName):
    with np.load(Path(dirPath) / activityFileName, allow_pickle=True) as f:
        x = f['x']
        trial_params = f['trial_params']
        model_output = f['model_output']
        model_state = f['model_state']
        mask = f.get('mask', None)

    choice12, choiceB, choiceLR, seqAB, loc12_label = derive_choice_labels_from_model_output(
        model_output,
        trial_params,
        mask=mask)

    qAs = np.array([trial_params[i]['qA'] for i in range(len(trial_params))])
    qBs = np.array([trial_params[i]['qB'] for i in range(len(trial_params))])

    return x, trial_params, model_state, choice12, choiceB, choiceLR, qAs, qBs, seqAB, loc12_label


In [ ]:
def regressBehavior(choiceB, qAs, qBs):
    idx = (qAs != 0) & (qBs != 0)
    X = np.log(qBs[idx] / qAs[idx]).reshape(-1, 1)
    y = choiceB[idx]
    model = LogisticRegression()
    model.fit(X, y)
    return model


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def plotBehavior(qAs, qBs, trial_params, choiceB, savePath=None):
    model = regressBehavior(choiceB, qAs, qBs)
    a0, a1 = model.intercept_[0], model.coef_[0][0]

    offerRatio = np.full(qAs.shape, np.nan, dtype=float)
    np.divide(qBs, qAs, out=offerRatio, where=(qAs != 0) & (qBs != 0))
    offerRatio[offerRatio > 10] = 10
    choiceTarget = np.array([trial_params[i]['chooseB'] for i in range(len(trial_params))], dtype=bool)

    oRs = np.unique(offerRatio[~np.isnan(offerRatio)])
    simulationChoice = np.zeros(oRs.shape)
    teachingChoice = np.zeros(oRs.shape)
    for ii, oR in enumerate(oRs):
        simulationChoice[ii] = np.mean(choiceB[offerRatio == oR])
        teachingChoice[ii] = np.mean(choiceTarget[offerRatio == oR])

    fitChoice = sigmoid(a0 + a1 * np.log(oRs)) * 100

    fig, ax = plt.subplots(figsize=(4, 3), dpi=150)
    ax.plot(oRs, teachingChoice * 100, 'o', label='Teaching', color='tab:orange', markerfacecolor='white')
    ax.plot(oRs, simulationChoice * 100, 'o', label='Simulation', color='teal', markerfacecolor='lightgray')
    ax.plot(oRs, fitChoice, label='Fit', color='teal')
    ax.legend(loc='upper left', frameon=True)
    ax.set_xlabel('Offer qB:qA')
    ax.set_ylabel('Choose B %')
    ax.set_xscale('log')
    ax.set_xticks([1/4, 1/2, 2/2, 3/2, 8/3, 8/1])
    ax.xaxis.set_minor_locator(plt.NullLocator())
    ax.set_xticklabels(['1:4', '1:2', '2:2', '3:2', '8:3', '8:1'])
    ax.set_ylim(-5, 105)
    ax.set_title(f'spatialTaskLateMapping fit ind. point={np.exp(-a0/a1):.2f}', fontsize=8)
    fig.tight_layout()
    if savePath is not None:
        fig.savefig(savePath)
    return fig, model


In [ ]:
if not modelDirs:
    raise FileNotFoundError('No spatialTaskLateMapping activitityTestGrid.npz files found. Run training_spatialTask.py first.')

dirPath = modelDirs[0]
x, trial_params, model_state, choice12, choiceB, choiceLR, qAs, qBs, seqAB, loc12_label = importAndPreprocess(dirPath)

singleFigurePath = figureRoot / 'Fig1D_behavior_spatialTaskLateMapping_single.pdf'
fig, model = plotBehavior(qAs, qBs, trial_params, choiceB, singleFigurePath)
print('dirPath:', dirPath)
print('saved:', singleFigurePath)
print('fit intercept:', model.intercept_[0])
print('fit coef:', model.coef_[0][0])
print('fit indifference point:', np.exp(-model.intercept_[0] / model.coef_[0][0]))
